# Báo cáo Project
Lớp TTNT-T20242-156727, Nhóm G17

## 1. Thông tin chung

### Thành viên
- Nguyễn Khánh Toàn 20225936
- Phạm Quốc Cường 20225604
- Nguyễn Bùi Tuấn Linh 20225732
- Hồ Tuấn Huy 20225856
- Hà Trung Chiến 20225794

### Lịch thực hiện
- W25: Đăng ký nhóm 
- W26: Đề xuất project (1/3)
- W31: Báo cáo tiến độ giữa kỳ (5/4)
- W37: Hoàn thành và gửi báo cáo project (17/5)
- W38-40: Trình bày project, Q&A

## 2. Đề xuất project (W26)

### Bài toán
#### Hệ thống gợi ý trang phục
 
Hình thức hoạt động: Người dùng đưa 1 bức ảnh về các trang phục như áo, quần, giày, ... và xong đó hệ thống sẽ đưa ra những bức ảnh giống về những bộ quần áo giống với đầu vào nhất

### Phương pháp
#### 1. Xử lý ảnh đầu vào
- Chuẩn hóa tập dữ liệu ảnh về dạng các ma trận và resize về kích thước chuẩn
- Chuẩn hóa giá trị pixel
- Thêm các dữ liệu từ các góc của bức ảnh (Augmentation Data)

#### 2. Trích xuất đặc trưng & lưu trữ dữ liệu
- Xây dựng mô hình CNN nhận diện các hình ảnh quần áo theo tên các nhóm ảnh quần áo
- Cắt các lớp cuối phân loại của mô hình, giữ lại các lớp trích xuất đặc trưng
- Lưu trước tập các vector đặc trưng của các bức ảnh quần áo

#### 3. Truy vấn & tìm kiếm ảnh tương tự
- Hệ thống trích xuất vector đặc trưng của ảnh mới được tải lên
- Sử dụng phương pháp tìm kiếm gần nhất (KNN – Nearest Neighbors) trong không gian vector để tìm các ảnh có đặc trưng gần nhất với ảnh đầu vào
- Lấy danh sách ảnh tương tự và hiển thị kết quả cho người dùng

### Phân công
- Nguyễn Khánh Toàn: Tìm hiểu về mạng CNN (đặc biệt là lớp Convolution) và các lớp trích xuất đặc trưng của mô hình 
- Phạm Quốc Cường: Tìm hiểu về các thuật toán cập nhật tham số
- Nguyễn Bùi Tuấn Linh: Tìm hiểu về mạng nơ ron nhân tạo
- Hồ Tuấn Huy: Tìm hiểu về các lớp còn lại trong mạng phân loại các bức ảnh (Max  Pooling, Batchnorm, ...)
- Hà Trung Chiến: Thu thập dữ liệu và tìm hiểu các phương pháp xử lý dữ liệu

## 3. Tiến độ giữa kỳ (W31)

### Chương trình
- Các bức ảnh được resize về chuẩn đầu vào model với kích thước 224x224 và đưa về khoảng giá trị từ 0 đến 1, có kì vọng và độ lệch chuẩn cho từng lớp theo tứ tự Số lớp x Chiều cao x Chiều rộng là 
[0.485, 0.456, 0.406] và [0.229, 0.224, 0.225]

In [ ]:
transform = Compose(
        [
            Resize((224, 224)),
            ToTensor(),
            Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ]
    )

 - Sử dụng mô hình resnet50 cho quá trình phân loại ảnh và trích xuất ra các đặc trưng của ảnh
 ![Resnet50](https://towardsdatascience.com/wp-content/uploads/2022/08/0tH9evuOFqk8F41FG.png)
 - Bộ dataset cho việc training được thu thập từ 2 bộ dữ liệu chính là bộ dữ liệu [outfit-items](https://www.kaggle.com/datasets/kritanjalijain/outfititems) và bộ [clothes-dataset](https://universe.roboflow.com/sookmyung-women-university/clothes-dataset-e9wyj) đều đã được public và cho phép sử dụng không cho mục đích thương mại
- Sau đó lấy 2 giá trị một là nhãn dự đoán của mô hình và vector chứa dữ liệu đặc trưng của bức ảnh sau khi được làm phẳng:

In [ ]:
def forward(self, x):
    x = self.conv0(x)
    x = self.max_pool(x)

    x = self.conv1(x)
    x = self.conv2(x)
    x = self.conv3(x)
    x = self.conv4(x)

    x = self.avg_pool(x)
    avg = self.flatten(x)
    x = self.ffn(avg)

    return x, avg


- Trong quá trình dự đoán, nhãn của bức ảnh sẽ quyết định bức ảnh thuộc về lớp nào nhằm giảm thời gian tìm kiếm ảnh tương đồng do giới hạn được lớp ảnh
- Sử dụng thuật toán KNN lấy k hàng xóm có độ đo cosine Similarity cao nhất làm kết quả dự đoán ảnh tương đồng cho ảnh đầu vào

In [ ]:
import numpy as np

def top_k_cosine_indices(vector, cursor, k=5):
    vector = np.array(vector)
    list_of_vectors = [doc["feature"] for doc in cursor]
    names = [doc["name"] for doc in cursor]
    matrix = np.array(list_of_vectors) 

    dot_product = np.dot(matrix, vector) 
    norm_matrix = np.linalg.norm(matrix, axis=1)  
    norm_vector = np.linalg.norm(vector)

    cosine_similarities = dot_product / (norm_matrix * norm_vector)  

    top_k_indices = np.argsort(cosine_similarities)[-k:][::-1]

    return [names[i] for i in top_k_indices]

### Kết quả, vấn đề gặp phải
#### Kết quả
 - Kết quả training model phân loại:
    - Các thông số training :
        - loss: SGD 
        - Learning rate = 1e-3
        - Epochs: 50
    - Kết quả:
    ```bash
    Epoch 48/50. Iteration 121/121. Loss 0.223: 100%|█████| 121/121 [03:22<00:00,  1.67s]
    val_acc:  0.75

    Epoch 49/50. Iteration 121/121. Loss 0.284: 100%|█████| 121/121 [03:21<00:00,  1.67s]
    val_acc:  0.8911

    Epoch 50/50. Iteration 121/121. Loss 0.203: 100%|█████| 121/121 [03:22<00:00,  1.67s]
    val_acc:  0.9005
    ```
-  Demo kết quả sau bước sử dụng KNN dự đoán hàng xóm
    - Ảnh đầu vào :
        - <img src="Resoruces\Screenshot 2025-03-23 171636.png" alt="Hình ảnh minh họa" style="width: 300px; border-radius: 10px;">

    - Đầu ra dự đoán
    <table>
    <tr>
        <td><img src="Resoruces\Screenshot 2025-03-23 171705.png" width="200"/></td>
        <td><img src="Resoruces\Screenshot 2025-03-23 171711.png" width="200"/></td>
        <td><img src="Resoruces\Screenshot 2025-03-23 171715.png" width="200"/></td>
        <td><img src="Resoruces\Screenshot 2025-03-23 171720.png" width="200"/></td>
    </tr>
    </table>

## 4. Cập nhật kết quả cuối kỳ (W37)

### Chi tiết phương pháp, dữ liệu 
....

### Chương trình
...

### Phân tích, đánh giá kết quả
...

### Cập nhật phân công, khối lượng công việc
<!-- công việc của các thành viên, tỷ lệ đóng góp của các thành viên -->
...